<a href="https://colab.research.google.com/github/BuruhArloji/PythonDataScienceHandbook/blob/master/Forecast_SKU_Canonical_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Forecast SKU Canonical Pipeline

Notebook ini menyiapkan data actual sales supaya SKU yang berubah kode atau penulisan ukuran tetap bisa dibaca sebagai satu SKU historis yang konsisten.

Fokus output:
- load `Sales 2025` dan `Sales 2026`
- standardisasi kolom menjadi schema yang sama
- buat `canonical_sku` berbasis atribut produk, bukan kode SKU mentah
- audit kandidat SKU yang berubah kode/deskripsi/ukuran
- review khusus SKU yang muncul/berubah di awal 2025
- export table mapping/QC untuk disetujui sebelum dipakai forecast

Catatan source: path PT AJEINDONESIA yang diberikan saat inspeksi masih berupa OneDrive reparse/placeholder dan belum bisa dibaca oleh runtime. Notebook ini tetap menaruh path tersebut sebagai prioritas, lalu fallback ke salinan Ajethai yang bisa dibaca.

In [2]:
from pathlib import Path
import os
import re
import zipfile
import warnings
from google.colab import drive
import numpy as np
import pandas as pd

In [4]:
try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ModuleNotFoundError:
    HAS_MATPLOTLIB = False
    plt = None

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

In [5]:
drive.mount("/content/drive")

Mounted at /content/drive


In [8]:
SALES_FILE =  "/content/drive/MyDrive/Data Sales/Indonesia Sales Dashboard 2026.xlsm"

## 1. Load Source Sheets

Workbook ini punya dua sheet actual utama untuk penyelarasan:
- `Sales 2025`: actual historical 2025, header di row 1
- `Sales 2026`: actual 2026, header di row 1 dengan beberapa kolom kosong di awal

In [9]:
sales_2025_raw = pd.read_excel(SALES_FILE, sheet_name="Sales 2025", engine="openpyxl")
sales_2026_raw = pd.read_excel(SALES_FILE, sheet_name="Sales 2026", engine="openpyxl")

print("Sales 2025:", sales_2025_raw.shape)
print("Sales 2026:", sales_2026_raw.shape)

display(sales_2025_raw.head(3))
display(sales_2026_raw.head(3))

Sales 2025: (20354, 36)
Sales 2026: (16938, 33)


,ROW 2,Year,Month,Day,Channel Group,Branch,Channel,Region,Cust Code,Customer Name,District,Brand,Format,Flavor,Short Item Description (Edit),Item Code (Edited),Box Content,ID Sales,CU,CF,imp_paquetevta,imp_dscto,imp_cobrarvta,imp_impuesto2,imp_netovta,Date,Month2,Channel Group2,Distribution Point,Region_Cust Group,Trading Terms,Net Sales (Adjusted),List Expense,Column1,Item Code (Real),Short Item Description (Edit).1
0,1.0,2025,January,2,CEDIS,FACTORY,RETAIL,GREATER JAKARTA,413970,WARUNG STEAK & SHAKE_JATIWARINGIN ...,BEKASI,BIG,1.625,LIME,BIG LIME NRP 1Lt x6,523884,6,NaN,1.625,5,328500.0,0.0,328500.0,32554.054,295945.946,2025-01-02,1,FACTORY SALES,FACTORY,FACTORY SALES,0.0,295945.946,NaN,FXC,523417,BIG LIME NRP 1625 ML 6
1,2.0,2025,January,2,CEDIS,FACTORY,RETAIL,GREATER JAKARTA,414031,WAROENG STEAK & SHAKE_RAWAMANGUN ...,JAKARTA TIMUR,BIG,1.625,LIME,BIG LIME NRP 1Lt x6,523884,6,NaN,1.950,6,394200.0,0.0,394200.0,39064.865,355135.135,2025-01-02,1,FACTORY SALES,FACTORY,FACTORY SALES,0.0,355135.135,NaN,FXC,523417,BIG LIME NRP 1625 ML 6
2,3.0,2025,January,2,CEDIS,FACTORY,RETAIL,GREATER JAKARTA,414033,WAROENG STEAK & SHAKE_KALIMALANG ...,JAKARTA TIMUR,BIG,1.625,LIME,BIG LIME NRP 1Lt x6,523884,6,NaN,1.300,4,262800.0,0.0,262800.0,26043.243,236756.757,2025-01-02,1,FACTORY SALES,FACTORY,FACTORY SALES,0.0,236756.757,NaN,FXC,523417,BIG LIME NRP 1625 ML 6


,Unnamed: 0,Unnamed: 1,ROW 2,fecha_liquidacion,Year of fecha_liquidacion,Month of fecha_liquidacion,Day of fecha_liquidacion,DescCanalLocal (grupo),desc_sucursal,DescCanalLocal,desc_region,cod_cliente,nomb_cliente,Region_CustGroup,Desc_SubCanal,desc_provincia,desc_marca,desc_formato,desc_sabor,desc_articulo_corto,cod_articulo,cant_contenido,id_venta,CU,CF,imp_paquetevta,imp_dscto,imp_cobrarvta,imp_impuesto2,imp_netovta,Month,Year,Full Price
0,JAN,NaN,1,2026-01-02,2026,January,2,MODERN,LOGISTIC PROVIDER BEKASI,SUPERMARKET,MODERN TRADE,417256,PT.TIP TOP_PONDOK GEDE ...,TIP TOP,SUPERMERCADOS,BEKASI,BIG,0.4,COLA,BIG COLA 400ML x12,500040,12,ASIA|0088|30|02|FXC|1334,0.32,2,83640.0,836.4,82803.6,8205.76,74597.84,1,2026,41820.0
1,NaN,NaN,2,2026-01-02,2026,January,2,MODERN,LOGISTIC PROVIDER BEKASI,SUPERMARKET,MODERN TRADE,417256,PT.TIP TOP_PONDOK GEDE ...,TIP TOP,SUPERMERCADOS,BEKASI,BIG,0.4,LIME,BIG LIME NRP 400 ML 12,523410,12,ASIA|0088|30|02|FXC|1334,0.32,2,83640.0,836.4,82803.6,8205.76,74597.84,1,2026,41820.0
2,NaN,NaN,3,2026-01-02,2026,January,2,MODERN,LOGISTIC PROVIDER BEKASI,SUPERMARKET,MODERN TRADE,417256,PT.TIP TOP_PONDOK GEDE ...,TIP TOP,SUPERMERCADOS,BEKASI,BIG,0.4,STRAWBERRY,BIG SBRRY NRP 400 ML 12,523414,12,ASIA|0088|30|02|FXC|1334,1.60,10,418200.0,4182.0,414018.0,41028.81,372989.19,1,2026,41820.0


## 2. Standardisasi Actual Sales

Tujuannya membuat `Sales 2025` dan `Sales 2026` memiliki nama kolom seragam.

`canonical_sku` dibuat dari atribut produk yang lebih stabil:
`BRAND | FORMAT | FLAVOR | UNITS`.

Dengan cara ini, forecast bisa melihat continuity meskipun `Item Code/COD SKU` berubah.

In [ ]:
def clean_text(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().upper()
    value = re.sub(r"\s+", " ", value)
    value = value.replace("STRAWBERRY", "SBRRY")
    value = value.replace("NIPIS DAN MADU LIME HONEY", "NIPIS MADU")
    value = value.replace("NIPIS MADU LIME HONEY", "NIPIS MADU")
    return value

def norm_sku(value):
    if pd.isna(value):
        return ""
    value = str(value).strip()
    if value.endswith(".0"):
        value = value[:-2]
    return value

def norm_number(value):
    if pd.isna(value) or value == "":
        return np.nan
    try:
        return float(str(value).replace(",", "."))
    except Exception:
        return np.nan

def norm_format(value):
    n = norm_number(value)
    if pd.isna(n):
        return clean_text(value)
    return f"{n:g}"

def canonical_from_parts(brand, fmt, flavor, units):
    return "|".join([clean_text(brand), norm_format(fmt), clean_text(flavor), norm_sku(units)])

def standardize_sales_2025(df):
    out = pd.DataFrame(index=df.index)
    out["source_sheet"] = "Sales 2025"
    out["date"] = pd.to_datetime(df["Date"], errors="coerce")
    out["year"] = pd.to_numeric(df["Year"], errors="coerce")
    out["month"] = pd.to_numeric(df["Month2"], errors="coerce")
    out["channel_group"] = df["Channel Group2"].map(clean_text)
    out["channel"] = df["Channel"].map(clean_text)
    out["branch"] = df["Branch"].map(clean_text)
    out["region"] = df["Region"].map(clean_text)
    out["customer_code"] = df["Cust Code"].map(norm_sku)
    out["customer_name"] = df["Customer Name"].map(clean_text)
    out["customer_group"] = df["Region_Cust Group"].map(clean_text)
    out["brand"] = df["Brand"].map(clean_text)
    out["format"] = df["Format"].map(norm_format)
    out["flavor"] = df["Flavor"].map(clean_text)
    out["sku_desc"] = df["Short Item Description"].map(clean_text)
    out["sku_code"] = df["Item Code"].map(norm_sku)
    out["units"] = df["Box Content"].map(norm_sku)
    out["cu"] = pd.to_numeric(df["CU"], errors="coerce")
    out["cf"] = pd.to_numeric(df["CF"], errors="coerce")
    out["gross_sales"] = pd.to_numeric(df["imp_paquetevta"], errors="coerce")
    out["discount"] = pd.to_numeric(df["imp_dscto"], errors="coerce")
    out["net_sales"] = pd.to_numeric(df["imp_netovta"], errors="coerce")
    out["net_sales_adjusted"] = pd.to_numeric(df["Net Sales (Adjusted)"], errors="coerce")
    return out

def standardize_sales_2026(df):
    out = pd.DataFrame(index=df.index)
    out["source_sheet"] = "Sales 2026"
    out["date"] = pd.to_datetime(df["fecha_liquidacion"], errors="coerce")
    out["year"] = pd.to_numeric(df["Year"], errors="coerce")
    out["month"] = pd.to_numeric(df["Month"], errors="coerce")
    out["channel_group"] = df["DescCanalLocal (grupo)"].map(clean_text)
    out["channel"] = df["DescCanalLocal"].map(clean_text)
    out["branch"] = df["desc_sucursal"].map(clean_text)
    out["region"] = df["desc_region"].map(clean_text)
    out["customer_code"] = df["cod_cliente"].map(norm_sku)
    out["customer_name"] = df["nomb_cliente"].map(clean_text)
    out["customer_group"] = df["Region_CustGroup"].map(clean_text)
    out["brand"] = df["desc_marca"].map(clean_text)
    out["format"] = df["desc_formato"].map(norm_format)
    out["flavor"] = df["desc_sabor"].map(clean_text)
    out["sku_desc"] = df["desc_articulo_corto"].map(clean_text)
    out["sku_code"] = df["cod_articulo"].map(norm_sku)
    out["units"] = df["cant_contenido"].map(norm_sku)
    out["cu"] = pd.to_numeric(df["CU"], errors="coerce")
    out["cf"] = pd.to_numeric(df["CF"], errors="coerce")
    out["gross_sales"] = pd.to_numeric(df["imp_paquetevta"], errors="coerce")
    out["discount"] = pd.to_numeric(df["imp_dscto"], errors="coerce")
    out["net_sales"] = pd.to_numeric(df["imp_netovta"], errors="coerce")
    out["net_sales_adjusted"] = out["net_sales"]
    return out

actual = pd.concat(
    [standardize_sales_2025(sales_2025_raw), standardize_sales_2026(sales_2026_raw)],
    ignore_index=True,
)
actual = actual.dropna(subset=["date"])
actual["canonical_sku"] = [
    canonical_from_parts(b, f, t, u)
    for b, f, t, u in zip(actual["brand"], actual["format"], actual["flavor"], actual["units"])
]
actual["month_start"] = actual["date"].dt.to_period("M").dt.to_timestamp()
actual["week_start"] = actual["date"].dt.to_period("W-MON").apply(lambda p: p.start_time)
actual["size_liters_per_unit"] = pd.to_numeric(actual["format"], errors="coerce")
actual["liters_est"] = actual["cu"] * actual["size_liters_per_unit"]

print(actual.shape)
display(actual.head())
display(actual[["source_sheet", "sku_code", "canonical_sku", "sku_desc", "cu", "net_sales_adjusted"]].head(10))

## 3. SKU Canonical Audit

Bagian ini menjawab pertanyaan: SKU mana yang perlu disamakan sebelum forecast?

Rules:
- Jika satu `canonical_sku` punya lebih dari satu kode SKU, berarti kode berubah/bercabang.
- Jika satu kode SKU punya lebih dari satu `canonical_sku`, berarti kode tersebut tidak aman dipakai sendirian.
- `recommended_forecast_sku_code` dipilih dari kode yang muncul paling baru/terbesar kontribusinya di actual.

In [ ]:
actual_sku = (
    actual.groupby(["canonical_sku", "sku_code", "brand", "format", "flavor", "units", "sku_desc"], dropna=False)
    .agg(
        first_date=("date", "min"),
        last_date=("date", "max"),
        rows=("sku_code", "size"),
        cu=("cu", "sum"),
        net_sales=("net_sales_adjusted", "sum"),
    )
    .reset_index()
)

sku_pool = pd.concat(
    [
        actual_sku.assign(source_priority=1, source_type="actual")[["canonical_sku", "sku_code", "source_priority", "source_type", "last_date", "net_sales"]],
    ],
    ignore_index=True,
)

recommended = (
    sku_pool.sort_values(["canonical_sku", "source_priority", "last_date", "net_sales"], ascending=[True, False, False, False])
    .drop_duplicates("canonical_sku")
    .rename(columns={"sku_code": "recommended_forecast_sku_code"})
    [["canonical_sku", "recommended_forecast_sku_code"]]
)

canonical_audit = (
    actual_sku.groupby("canonical_sku", as_index=False)
    .agg(
        actual_sku_codes=("sku_code", lambda s: ", ".join(sorted(set(filter(None, s))))),
        actual_code_count=("sku_code", "nunique"),
        actual_desc_count=("sku_desc", "nunique"),
        first_actual=("first_date", "min"),
        last_actual=("last_date", "max"),
        actual_cu=("cu", "sum"),
        actual_net_sales=("net_sales", "sum"),
    )
    .merge(
        actual_sku.groupby("canonical_sku", as_index=False).agg(
            source_desc_examples=("sku_desc", lambda s: " || ".join(sorted(set(filter(None, s)))[:5])),
        ),
        on="canonical_sku",
        how="left",
    )
    .merge(recommended, on="canonical_sku", how="left")
)

canonical_audit["needs_review"] = (
    canonical_audit["actual_code_count"].fillna(0).gt(1)
    | canonical_audit["actual_desc_count"].fillna(0).gt(1)
)
canonical_audit = canonical_audit.sort_values(["needs_review", "actual_net_sales"], ascending=[False, False])

code_collision = (
    actual[["sku_code", "canonical_sku", "brand", "format", "flavor", "units", "source_sheet"]]
    .dropna(subset=["sku_code"])
    .query("sku_code != ''")
    .groupby("sku_code", as_index=False)
    .agg(
        canonical_count=("canonical_sku", "nunique"),
        canonical_skus=("canonical_sku", lambda s: " || ".join(sorted(set(s)))),
        sources=("source_sheet", lambda s: ", ".join(sorted(set(str(v) for v in s if pd.notna(v))))),
    )
    .query("canonical_count > 1")
    .sort_values("canonical_count", ascending=False)
)

print("Canonical SKU count:", canonical_audit.shape[0])
print("Needs review:", int(canonical_audit["needs_review"].sum()))
print("Code collisions:", code_collision.shape[0])
display(canonical_audit.head(30))
display(code_collision.head(30))

## 4. Review Perubahan Awal 2025

Bagian ini memisahkan SKU yang muncul di Jan-Mar 2025 dan membandingkannya dengan Apr-Dec 2025 serta 2026/Plan.

In [ ]:
actual["period_bucket"] = np.select(
    [
        (actual["date"] >= "2025-01-01") & (actual["date"] < "2025-04-01"),
        (actual["date"] >= "2025-04-01") & (actual["date"] < "2026-01-01"),
        actual["date"] >= "2026-01-01",
    ],
    ["2025 Jan-Mar", "2025 Apr-Dec", "2026 Actual"],
    default="Other",
)

period_sku = (
    actual.groupby(["period_bucket", "canonical_sku", "sku_code"], dropna=False)
    .agg(rows=("sku_code", "size"), cu=("cu", "sum"), net_sales=("net_sales_adjusted", "sum"))
    .reset_index()
)

period_matrix = (
    period_sku.pivot_table(
        index="canonical_sku",
        columns="period_bucket",
        values="cu",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
    .merge(canonical_audit[["canonical_sku", "actual_sku_codes", "source_desc_examples", "recommended_forecast_sku_code", "needs_review"]], on="canonical_sku", how="left")
)

for col in ["2025 Jan-Mar", "2025 Apr-Dec", "2026 Actual"]:
    if col not in period_matrix.columns:
        period_matrix[col] = 0

period_matrix["early_2025_only"] = period_matrix["2025 Jan-Mar"].gt(0) & period_matrix["2025 Apr-Dec"].eq(0) & period_matrix["2026 Actual"].eq(0)
period_matrix["continues_after_early_2025"] = period_matrix["2025 Jan-Mar"].gt(0) & (period_matrix["2025 Apr-Dec"].gt(0) | period_matrix["2026 Actual"].gt(0))
period_matrix = period_matrix.sort_values(["early_2025_only", "2025 Jan-Mar"], ascending=[False, False])

display(period_matrix.head(50))

## 5. Build Aligned Actual Tables

Tabel utama:
- `daily_actual`
- `weekly_actual`
- `monthly_actual`

Gunakan `canonical_sku` sebagai grain produk utama. Gunakan `sku_code` hanya sebagai detail/audit.

In [ ]:
forecast_dims = ["channel_group", "customer_group", "canonical_sku"]

daily_actual = (
    actual.groupby(["date", *forecast_dims], dropna=False)
    .agg(
        cu=("cu", "sum"),
        cf=("cf", "sum"),
        liters_est=("liters_est", "sum"),
        gross_sales=("gross_sales", "sum"),
        discount=("discount", "sum"),
        net_sales=("net_sales_adjusted", "sum"),
        sku_codes=("sku_code", lambda s: ", ".join(sorted(set(filter(None, s))))),
    )
    .reset_index()
)

weekly_actual = (
    actual.groupby(["week_start", *forecast_dims], dropna=False)
    .agg(
        cu=("cu", "sum"),
        cf=("cf", "sum"),
        liters_est=("liters_est", "sum"),
        net_sales=("net_sales_adjusted", "sum"),
    )
    .reset_index()
)

monthly_actual = (
    actual.groupby(["month_start", *forecast_dims], dropna=False)
    .agg(
        cu=("cu", "sum"),
        cf=("cf", "sum"),
        liters_est=("liters_est", "sum"),
        net_sales=("net_sales_adjusted", "sum"),
    )
    .reset_index()
)

print("daily_actual  ", daily_actual.shape)
print("weekly_actual ", weekly_actual.shape)
print("monthly_actual", monthly_actual.shape)

display(monthly_actual.head())

## 6. Visual Diagnostics

Grafik ini untuk mengecek apakah canonical mapping membuat history lebih continuous. Kalau `matplotlib` belum tersedia, cell ini akan menampilkan tabel ringkas saja.

In [ ]:
top_skus = (
    monthly_actual.groupby("canonical_sku")["net_sales"]
    .sum()
    .sort_values(ascending=False)
    .head(8)
    .index
)

plot_data = monthly_actual[monthly_actual["canonical_sku"].isin(top_skus)].copy()
pivot = plot_data.pivot_table(index="month_start", columns="canonical_sku", values="net_sales", aggfunc="sum", fill_value=0)

if HAS_MATPLOTLIB:
    ax = pivot.plot(figsize=(14, 6), marker="o")
    ax.set_title("Monthly Net Sales by Canonical SKU - Top 8")
    ax.set_xlabel("Month")
    ax.set_ylabel("Net Sales")
    ax.grid(True, alpha=0.3)
    plt.legend(loc="center left", bbox_to_anchor=(1, 0.5))
    plt.tight_layout()
    plt.show()
else:
    print("matplotlib is not installed; showing monthly pivot preview instead.")
    display(pivot.tail(12))

review_plot = canonical_audit.head(20).copy()
review_plot["total_value"] = review_plot["actual_net_sales"].fillna(0)
if HAS_MATPLOTLIB:
    ax = review_plot.sort_values("total_value").plot.barh(x="canonical_sku", y="total_value", figsize=(12, 8), legend=False)
    ax.set_title("Top Review Candidates by Actual Value")
    ax.set_xlabel("Value")
    plt.tight_layout()
    plt.show()
else:
    display(review_plot[["canonical_sku", "actual_sku_codes", "source_desc_examples", "total_value", "needs_review"]])

## 7. Export Audit Tables

CSV ini bisa dipakai untuk review manual sebelum mapping dipakai permanen.

In [ ]:
exports = {
    "canonical_sku_audit.csv": canonical_audit,
    "sku_code_collision.csv": code_collision,
    "early_2025_review.csv": period_matrix,
    "daily_actual_forecast_ready.csv": daily_actual,
    "weekly_actual_forecast_ready.csv": weekly_actual,
    "monthly_actual_forecast_ready.csv": monthly_actual,
}

for name, df in exports.items():
    path = OUTPUT_DIR / name
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(path, df.shape)

## 8. Recommended Mapping Policy

Saran pemakaian:

1. Grain forecast utama: `channel_group + customer_group + canonical_sku`.
2. Jangan forecast berdasarkan `sku_code` mentah, karena kode historis antar tahun bisa berbeda.
3. Simpan `recommended_forecast_sku_code` hanya sebagai label output/reporting.
4. Review manual semua baris `needs_review = True`, khususnya SKU yang hanya muncul di awal 2025 atau kode yang collision.
5. Setelah disetujui, mapping ini bisa dijadikan master kecil: `canonical_sku`, `recommended_forecast_sku_code`, `brand`, `format`, `flavor`, `units`, `mapping_status`, `notes`.